# Day 02 Lab: Find and Fix the Bug

**Course:** ITSC 1213, Summer 2026
**Date:** Thursday, May 21, 2026
**Time:** 2:30 PM to 4:25 PM

## What you will do

In this lab you receive a small Java program called `GardenAssistant`.
It is **inspired by** the in-class `AnimalTrainerMatcher` demo, but
the domain is different (plants and gardeners instead of animals and
trainers) and there is **exactly one bug** somewhere in the code.

Your job is to:

1. Run the program as is and observe the output.
2. Compare the output to the **expected output** described in the
   comments of `main`.
3. Find the single line that contains the bug.
4. Fix it. The fix is one operator change.
5. Re-run and confirm the output now matches the expectation.

**Time budget:** 30 minutes.

## How this notebook runs Java

Colab does not run Java natively, so we use shell magics to:

1. Confirm that a JDK is installed in the Colab runtime.
2. Write the Java source to a file using `%%writefile`.
3. Compile with `javac`.
4. Run with `java`.

If a step fails, the notebook will tell you with an error message.

## Step 0: Check that Java is installed

Colab usually ships with a Java Development Kit. Run the cell below
and confirm you see a `javac` and a `java` version line. If the cell
says "command not found", uncomment the install line and run again.

In [13]:
# If javac is missing in your runtime, uncomment the apt-get line below.
# !apt-get install -y openjdk-17-jdk-headless > /dev/null 2>&1
!java --version
!javac --version

openjdk 17.0.18 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)
javac 17.0.18


## Step 1: Write the Java source to a file

Run the next cell. It uses the `%%writefile` magic to save the
Java source code into a file named `GardenAssistant.java` in the
Colab working directory.

**Do not edit the file inside this cell yet.** First run it as is,
see the buggy behavior, then come back and apply your fix.

In [21]:
%%writefile GardenAssistant.java
import java.util.*;

/**
 * GardenAssistant
 *
 * A small program that catalogs plants and matches each plant with the
 * gardener whose preferences best fit. Inspired by the in-class
 * AnimalTrainerMatcher demo but with a different domain so students
 * can practice the same patterns in a new setting.
 */
public class GardenAssistant {

    /**
     * Returns a hard-coded map of plants. Keys are the common names
     * of the plants. Values are maps of trait name to trait value.
     */
    public static Map<String, Map<String, Object>> getPlantMap() {
        Map<String, Map<String, Object>> plants = new HashMap<>();

        Map<String, Object> sunflower = new HashMap<>();
        sunflower.put("water_liters", 2.5);
        sunflower.put("type", "flower");
        sunflower.put("environment", "outdoor");
        sunflower.put("light_level", "high");
        sunflower.put("season", "summer");
        sunflower.put("edible", false);
        sunflower.put("height_cm", 200.0);
        sunflower.put("soil_pref", "loam");
        plants.put("sunflower", sunflower);

        Map<String, Object> basil = new HashMap<>();
        basil.put("water_liters", 1.0);
        basil.put("type", "herb");
        basil.put("environment", "indoor");
        basil.put("light_level", "high");
        basil.put("season", "summer");
        basil.put("edible", true);
        basil.put("height_cm", 30.0);
        basil.put("soil_pref", "loam");
        plants.put("basil", basil);

        Map<String, Object> fern = new HashMap<>();
        fern.put("water_liters", 3.0);
        fern.put("type", "fern");
        fern.put("environment", "indoor");
        fern.put("light_level", "low");
        fern.put("season", "year_round");
        fern.put("edible", false);
        fern.put("height_cm", 60.0);
        fern.put("soil_pref", "peat");
        plants.put("fern", fern);

        Map<String, Object> tomato = new HashMap<>();
        tomato.put("water_liters", 4.0);
        tomato.put("type", "vegetable");
        tomato.put("environment", "outdoor");
        tomato.put("light_level", "high");
        tomato.put("season", "summer");
        tomato.put("edible", true);
        tomato.put("height_cm", 150.0);
        tomato.put("soil_pref", "loam");
        plants.put("tomato", tomato);

        return plants;
    }

    /**
     * Returns a hard-coded map of gardeners. Each gardener has a set
     * of preferred trait values.
     */
    public static Map<String, Map<String, String>> getGardenerMap() {
        Map<String, Map<String, String>> gardeners = new HashMap<>();

        Map<String, String> bob = new HashMap<>();
        bob.put("type", "vegetable");
        bob.put("environment", "outdoor");
        bob.put("light_level", "high");
        bob.put("season", "summer");
        bob.put("soil_pref", "loam");
        gardeners.put("Bob", bob);

        Map<String, String> carol = new HashMap<>();
        carol.put("type", "fern");
        carol.put("environment", "indoor");
        carol.put("light_level", "low");
        carol.put("soil_pref", "peat");
        gardeners.put("Carol", carol);

        Map<String, String> dan = new HashMap<>();
        dan.put("type", "herb");
        dan.put("environment", "indoor");
        dan.put("light_level", "high");
        dan.put("soil_pref", "loam");
        gardeners.put("Dan", dan);

        return gardeners;
    }

    /**
     * Selects plants whose numeric trait is AT LEAST the given
     * threshold (the comparison is >=), or whose categorical trait
     * equals the given value.
     *
     * Example: selectByThreshold(plants, "water_liters", 2.5) should
     * return every plant whose water_liters value is 2.5 or more.
     */
    public static Map<String, Map<String, Object>> selectByThreshold(
            Map<String, Map<String, Object>> plants,
            String characteristic,
            Object threshold) {

        Map<String, Map<String, Object>> selected = new HashMap<>();

        for (String name : plants.keySet()) {
            Map<String, Object> traits = plants.get(name);
            Object value = traits.get(characteristic);

            if (value instanceof Number && threshold instanceof Number) {
                // Numeric branch.
                if (((Number) value).doubleValue() >=
                    ((Number) threshold).doubleValue()) {
                    selected.put(name, traits);
                }
            } else if (threshold instanceof String) {
                // Categorical branch.
                if (value != null && value.equals(threshold)) {
                    selected.put(name, traits);
                }
            }
        }

        return selected;
    }

    /**
     * Multiplies a numeric trait by a given factor for every plant.
     * Returns a new map; the original is not modified.
     */
    public static Map<String, Map<String, Object>> scaleResource(
            Map<String, Map<String, Object>> plants,
            String characteristic,
            double factor) {

        Map<String, Map<String, Object>> updated = new HashMap<>();
        for (String name : plants.keySet()) {
            Map<String, Object> traits = new HashMap<>(plants.get(name));
            Object value = traits.get(characteristic);
            if (value instanceof Number) {
                traits.put(characteristic,
                           ((Number) value).doubleValue() * factor);
            }
            updated.put(name, traits);
        }
        return updated;
    }

    /**
     * Adds a new plant to the collection and returns a new map.
     */
    public static Map<String, Map<String, Object>> addPlant(
            Map<String, Map<String, Object>> plants,
            String name,
            Map<String, Object> traits) {
        Map<String, Map<String, Object>> updated = new HashMap<>(plants);
        updated.put(name, traits);
        return updated;
    }

    /**
     * For each plant, picks the gardener whose preferences match the
     * largest number of the plant's traits.
     */
    public static Map<String, String> matchGardeners(
            Map<String, Map<String, Object>> plants,
            Map<String, Map<String, String>> gardeners) {
        Map<String, String> matches = new HashMap<>();
        for (String plant : plants.keySet()) {
            Map<String, Object> traits = plants.get(plant);
            int bestScore = -1;
            String bestGardener = null;
            for (String gardener : gardeners.keySet()) {
                Map<String, String> prefs = gardeners.get(gardener);
                int score = 0;
                for (String key : prefs.keySet()) {
                    if (traits.containsKey(key)
                        && traits.get(key).equals(prefs.get(key))) {
                        score++;
                    }
                }
                if (score > bestScore) {
                    bestScore = score;
                    bestGardener = gardener;
                }
            }
            matches.put(plant, bestGardener);
        }
        return matches;
    }

    /** Helper that prints the plants and their traits. */
    public static void reportPlants(
            Map<String, Map<String, Object>> plants) {
        System.out.println("Plant Report:");
        // Sort by name for stable output across JVM runs.
        List<String> names = new ArrayList<>(plants.keySet());
        Collections.sort(names);
        for (String name : names) {
            System.out.println(name + ":");
            Map<String, Object> traits = plants.get(name);
            List<String> keys = new ArrayList<>(traits.keySet());
            Collections.sort(keys);
            for (String key : keys) {
                System.out.println("  - " + key + " = " + traits.get(key));
            }
            System.out.println("//");
        }
    }

    /** Helper that prints the gardener matches. */
    public static void reportMatches(Map<String, String> matches) {
        System.out.println("Gardener Matches:");
        List<String> plants = new ArrayList<>(matches.keySet());
        Collections.sort(plants);
        for (String plant : plants) {
            System.out.println("- " + plant
                + " is best matched with " + matches.get(plant));
        }
    }

    public static void main(String[] args) {
        Map<String, Map<String, Object>> plants = getPlantMap();
        Map<String, Map<String, String>> gardeners = getGardenerMap();

        // Step 1: high-water plants (water_liters >= 2.5).
        // Expected (in alphabetical order): fern, sunflower, tomato.
        Map<String, Map<String, Object>> highWater =
            selectByThreshold(plants, "water_liters", 2.5);
        System.out.println("=== High-water plants (water_liters >= 2.5) ===");
        reportPlants(highWater);

        // Step 2: summer plants (categorical match).
        // Expected (alphabetical): basil, sunflower, tomato.
        Map<String, Map<String, Object>> summer =
            selectByThreshold(plants, "season", "summer");
        System.out.println("=== Summer plants ===");
        reportPlants(summer);

        // Step 3: scale water by 1.5 (no change to original map).
        Map<String, Map<String, Object>> watered =
            scaleResource(plants, "water_liters", 1.5);
        System.out.println("=== After scaling water by 1.5 ===");
        reportPlants(watered);

        // Step 4: add a new plant (orchid).
        Map<String, Object> orchid = new HashMap<>();
        orchid.put("water_liters", 0.5);
        orchid.put("type", "flower");
        orchid.put("environment", "indoor");
        orchid.put("light_level", "medium");
        orchid.put("season", "year_round");
        orchid.put("edible", false);
        orchid.put("height_cm", 40.0);
        orchid.put("soil_pref", "bark");
        plants = addPlant(plants, "orchid", orchid);

        // Step 5: gardener matches for the (now larger) plant set.
        Map<String, String> matches = matchGardeners(plants, gardeners);
        System.out.println("=== Gardener matches ===");
        reportMatches(matches);
    }
}


Overwriting GardenAssistant.java


## Step 2: Compile and run

Compile the file with `javac` and run it with `java`. If compilation
fails, you will see error lines starting with `GardenAssistant.java:`
and a line number.

In [22]:
!javac GardenAssistant.java

In [24]:
!java GardenAssistant

=== High-water plants (water_liters >= 2.5) ===
Plant Report:
fern:
  - edible = false
  - environment = indoor
  - height_cm = 60.0
  - light_level = low
  - season = year_round
  - soil_pref = peat
  - type = fern
  - water_liters = 3.0
//
sunflower:
  - edible = false
  - environment = outdoor
  - height_cm = 200.0
  - light_level = high
  - season = summer
  - soil_pref = loam
  - type = flower
  - water_liters = 2.5
//
tomato:
  - edible = true
  - environment = outdoor
  - height_cm = 150.0
  - light_level = high
  - season = summer
  - soil_pref = loam
  - type = vegetable
  - water_liters = 4.0
//
=== Summer plants ===
Plant Report:
basil:
  - edible = true
  - environment = indoor
  - height_cm = 30.0
  - light_level = high
  - season = summer
  - soil_pref = loam
  - type = herb
  - water_liters = 1.0
//
sunflower:
  - edible = false
  - environment = outdoor
  - height_cm = 200.0
  - light_level = high
  - season = summer
  - soil_pref = loam
  - type = flower
  - water_lite

## Step 3: Compare output to expectations

Look at the output above. The `main` method documents what each
section should print:

* **Step 1 in `main`:** "High-water plants (water_liters >= 2.5)".
  The expected plants, in alphabetical order, are **fern**,
  **sunflower**, and **tomato**, because their `water_liters` are
  3.0, 2.5, and 4.0 respectively (all 2.5 or more).
* **Step 2 in `main`:** "Summer plants". The expected plants are
  **basil**, **sunflower**, and **tomato**.
* **Step 3:** "After scaling water by 1.5". Each plant's water
  value should be 1.5 times what it was originally.
* **Step 4:** the orchid is added.
* **Step 5:** "Gardener matches". Each plant should land with the
  gardener whose preferences match the most of its traits.

Focus on the **Step 1 output** first. Does it match what is
expected? Whose plants are showing up? Whose plants are missing?

## Step 4: Find the bug

Read the source of `selectByThreshold` carefully. The Javadoc
comment above the method states the intended behavior. Compare
that with what the code actually does on the numeric branch.

**One single character is wrong.** Change it.

## Step 5: Apply your fix

Edit the cell from Step 1 to fix the bug. Then re-run:

1. The Step 1 cell (`%%writefile GardenAssistant.java`) so the
   updated source is saved.
2. The compile cell.
3. The run cell.

Confirm that the Step 1 output of `main` now lists **fern**,
**sunflower**, and **tomato**.

## Step 6: Reflect (write a short answer)

In a markdown cell or a comment, briefly answer:

1. Which line did you change, and what did you change it to?
2. Why did the original line produce the wrong output?
3. Was the categorical branch (Step 2 in `main`) affected by the
   bug? Why or why not?

Your reflection goes inside the notebook (you can add a markdown
cell below this one). When you submit on Canvas, your `.ipynb`
file should already contain the fix and the reflection.

_Add your reflection in a new markdown cell below this one._

1. I changed the "selectByThreshold()" method of water liters as the orginal had the wrong boolean "<" rather than ">="
2. The output the wrong answer as it cause the method to scan for low water plants rather than the expected high water plants (above 2.5 liters)
3. Step 2 is not affected as the code is modular, Step 1 is only affected by the water liter input while Step 2 is affected by the season input. Therefore category summer is not affect by the previous error

## Submission

Follow the instructions in `Instructions.md` (in this same Lab
folder) for packaging and uploading the completed notebook with
your quiz answers as a single ZIP on Canvas.